In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dataclasses import dataclass

from tqdm import tqdm

import orjson
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm, trange

from sae_java_bug.logger import logger
from sae_java_bug.sparse_autoencoders.utils import get_device


@dataclass
class VulnerabilityData:
    vulnerable: torch.Tensor
    secure: torch.Tensor


/Users/rmelo/miniconda3/envs/demeanor/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import glob
import orjson
import torch
from tqdm import tqdm
from sae_java_bug.sparse_autoencoders.sae_exploration import ActivationsSchema

from sae_java_bug.logger import logger
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import numpy as np
from typing import Dict, Tuple
from sklearn.model_selection import KFold

class MLP(nn.Module):
    def __init__(
        self, input_dim, output_dim=2, hidden=(128, 64), dropout=0.3, use_bn=True
    ):
        super().__init__()
        h1, h2 = hidden
        self.use_bn = use_bn

        self.fc1 = nn.Linear(input_dim, h1)
        self.bn1 = nn.BatchNorm1d(h1) if use_bn else nn.Identity()
        self.fc2 = nn.Linear(h1, h2)
        self.bn2 = nn.BatchNorm1d(h2) if use_bn else nn.Identity()
        self.fc3 = nn.Linear(h2, output_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.drop(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.drop(x)

        x = self.fc3(x)  # logits
        return x

def evaluate(loader, model, device=get_device(verbose=False)):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())
    y_true = torch.cat(all_labels)
    y_pred = torch.cat(all_preds)
    return accuracy_score(y_true, y_pred), y_true, y_pred

def train_small_mlp_kfold(class_negative: torch.Tensor, class_positive: torch.Tensor, k=5):

    # Create labels
    vuln_labels = torch.ones(class_positive.size(0), dtype=torch.float32)
    safe_labels = torch.zeros(class_negative.size(0), dtype=torch.float32)

    all_features = torch.cat((class_positive, class_negative), dim=0).float()
    all_labels = torch.cat((vuln_labels, safe_labels), dim=0).long()

    # print("All Features Shape:", all_features.shape)
    # print("All Labels Shape:", all_labels.shape)

    dataset = TensorDataset(all_features, all_labels)

    # Initialize KFold
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    fold = 1
    fold_acc = []

    for train_idx, test_idx in kf.split(all_features):

        # print(f"\n===== Fold {fold}/{k} =====")
        fold += 1

        train_subset = torch.utils.data.Subset(dataset, train_idx)
        test_subset  = torch.utils.data.Subset(dataset, test_idx)

        train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, drop_last=True)
        test_loader  = DataLoader(test_subset, batch_size=1000, shuffle=False, drop_last=False)

        # new model per fold
        model = MLP(input_dim=all_features.size(1)).to(device)
        # activate gradients
        model.train()
        torch.set_grad_enabled(True)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

        num_epochs = 50  # often fewer needed

        for epoch in range(num_epochs):
            model.train()
            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)

                optimizer.zero_grad()
                logits = model(x_batch)
                loss = F.cross_entropy(logits, y_batch)
                loss.backward()
                optimizer.step()

        # evaluate this fold
        acc, _, _ = evaluate(test_loader, model, device)
        fold_acc.append(acc)

    #     print(f"Fold Test Accuracy: {acc:.4f}")

    # print("\n===== K-Fold Results =====")
    # print(f"Fold Accuracies: {fold_acc}")
    # print(f"Mean Accuracy: {sum(fold_acc)/len(fold_acc):.4f}")

    return sum(fold_acc)/len(fold_acc)


device = get_device(verbose=True)

layer_scores = {}
RUN_ID = "run_20260127_044329"
RUN_ID = "run_20260127_195450" # this is the swapped operands activations
RUN_ID = "run_20260127_202722" # this is the cwes data
RUN_ID = "run_20260128_184854" # Trained with DeltaSeccommits


files = sorted(glob.glob(f"../artifacts/activations/{RUN_ID}/activations_layer_*"), key=lambda x: int(x.split("layer_")[1].split("_")[0]))

@dataclass
class VulnerabilityData:
    vulnerable: torch.Tensor
    secure: torch.Tensor


def compute_lda_fisher_scores(secure_features: torch.Tensor, vulnerable_features: torch.Tensor) -> float:
    # Create labels
    vuln_labels = torch.ones(vulnerable_features.size(0), dtype=torch.float32)
    safe_labels = torch.zeros(secure_features.size(0), dtype=torch.float32)
    all_features = torch.cat((vulnerable_features, secure_features), dim=0).float()
    all_labels = torch.cat((vuln_labels, safe_labels), dim=0).long()

    # LDA — build dataset for layer
    X = all_features.numpy()
    y = all_labels.numpy()

    # Train LDA
    lda = LinearDiscriminantAnalysis(n_components=1)
    proj = lda.fit_transform(X, y)

    vuln_proj = proj[: len(vulnerable_features)]
    safe_proj = proj[len(vulnerable_features):]

    mean_diff = vuln_proj.mean() - safe_proj.mean()
    var_sum = vuln_proj.var() + safe_proj.var()
    fisher_score = (mean_diff ** 2) / (var_sum + 1e-8)

    return float(fisher_score)


2026-01-28 19:28:15 [info     ] Getting device.                device=mps
2026-01-28 19:28:16 [info     ] Getting device.                device=mps


In [4]:


FisherScoreMap: Dict[Tuple[str, str], float] = {}
ProbingDataMap: Dict[Tuple[str, str], float] = {}
from typing import List
from collections import defaultdict

all_vulnerable_features: Dict[str, List[float]] = defaultdict(list)
all_safe_features : Dict[str, List[float]] = defaultdict(list)

for activation_file in files:
    logger.info("Processing file.", file=activation_file)

    cwe_tensor_map: Dict[str, VulnerabilityData] = {}
    vulnerable_features = torch.tensor([])
    safe_features = torch.tensor([])
    
    # Loads activations.
    with open(activation_file, "r") as f: 
        for line in tqdm(f, desc="Loading", unit=" lines", total=2500):
            try:
                data = orjson.loads(line)
            except orjson.JSONDecodeError as e:
                logger.error(f"JSON decoding error: {e}")
                # skip to the next file
                break
            
            cwe = data["cwe"]
            if cwe not in cwe_tensor_map:
                cwe_tensor_map[cwe] = VulnerabilityData(
                    vulnerable=torch.tensor([]),
                    secure=torch.tensor([]),
                )

            activations = ActivationsSchema(**data)
            vuln_tensor = torch.tensor(activations.vulnerable)
            safe_tensor = torch.tensor(activations.secure)


            vulnerable_features = torch.cat(
                (vulnerable_features, vuln_tensor.unsqueeze(0)), dim=0
            )
            safe_features = torch.cat((safe_features, safe_tensor.unsqueeze(0)), dim=0)
            

            cwe_tensor_map[cwe] = VulnerabilityData(
                vulnerable=torch.cat(
                    (cwe_tensor_map[cwe].vulnerable, vuln_tensor.unsqueeze(0)), dim=0
                ),
                secure=torch.cat(
                    (cwe_tensor_map[cwe].secure, safe_tensor.unsqueeze(0)), dim=0
                ),
            )

        print("Vulnerable Features Shape:", vulnerable_features.shape , "Safe Features Shape:", safe_features.shape)
        print("CWE:", cwe, "Vulnerable Shape:", vulnerable_features.shape, "Safe Shape:", safe_features.shape)
    
        all_vulnerable_features[
            "layer_" + activation_file.split("layer_")[1].split("_")[0]
        ] = torch.tensor(vulnerable_features)
        all_safe_features[
            "layer_" + activation_file.split("layer_")[1].split("_")[0]
        ]= torch.tensor(safe_features)
        

    # Save for later reuse
    layer_id = activation_file.split("layer_")[1].split("_")[0]
    torch.save(vulnerable_features, f"../artifacts/activations/{RUN_ID}/vulnerable_layer_{layer_id}.pt")
    torch.save(safe_features, f"../artifacts/activations/{RUN_ID}/safe_layer_{layer_id}.pt")

2026-01-28 19:28:16 [info     ] Processing file.               file=../artifacts/activations/run_20260128_184854/activations_layer_0_sae_blocks.0.hook_resid_post_component_hook_resid_post.hook_sae_acts_post.jsonl


Loading:  20%|█▉        | 489/2500 [00:09<01:11, 28.28 lines/s] 

2026-01-28 19:28:26 [error    ] JSON decoding error: unexpected end of data: line 1 column 79820 (char 79819)


Loading:  20%|█▉        | 491/2500 [00:09<00:38, 52.20 lines/s]
/var/folders/s4/2zl3nz4j225gfbx8fkbdv9sm0000gq/T/ipykernel_6340/3811333901.py:58: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ] = torch.tensor(vulnerable_features)
/var/folders/s4/2zl3nz4j225gfbx8fkbdv9sm0000gq/T/ipykernel_6340/3811333901.py:61: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ]= torch.tensor(safe_features)


Vulnerable Features Shape: torch.Size([491, 16384]) Safe Features Shape: torch.Size([491, 16384])
CWE: CWE-89 Vulnerable Shape: torch.Size([491, 16384]) Safe Shape: torch.Size([491, 16384])


In [6]:
for key, data in tqdm(cwe_tensor_map.items()):
    try:
        # Compute LDA Fisher Score for this layer and CWE
        fisher_score = compute_lda_fisher_scores(data.secure, data.vulnerable)
        # logger.info(f"  LDA Fisher Score for CWE {key}: {fisher_score:.4f}")
        FisherScoreMap[(layer_id, key)] = fisher_score
        probe_score = train_small_mlp_kfold(data.secure, data.vulnerable, k=5)
        ProbingDataMap[(layer_id, key)] = probe_score
    except Exception as e:
        logger.error(f"Error processing CWE {key} in layer {layer_id}: {e}")


fisher_score = compute_lda_fisher_scores(safe_features, vulnerable_features)
FisherScoreMap[(layer_id, "ALL")] = fisher_score

 91%|█████████ | 20/22 [00:23<00:01,  1.78it/s]

2026-01-28 19:29:32 [error    ] Error processing CWE CWE-415 in layer 0: The number of samples must be more than the number of classes.


100%|██████████| 22/22 [00:23<00:00,  1.05s/it]


In [ ]:
files = sorted(glob.glob(f"../artifacts/activations/{RUN_ID}/activations_layer_*"))

def get_data(files: list, layer_id: int, cwe: str):
    # Read file and extract data for given layer_id and cwe
    for file in files:
        if f"layer_{layer_id}_" in file:
            activation_file = file
            break
    else:
        raise ValueError(f"Layer {layer_id} not found in files.")
    

    cwe_tensor_map: Dict[str, VulnerabilityData] = {}
    vulnerable_features = torch.tensor([])
    safe_features = torch.tensor([])
    # Loads activations.
    with open(activation_file, "r") as f:
        for line in tqdm(f, desc="Loading", unit=" lines", total=2500):
            try:
                data = orjson.loads(line)
            except orjson.JSONDecodeError as e:
                logger.error(f"JSON decoding error: {e}")
                # skip to the next file
                break
            _cwe = data["cwe"]
            if _cwe != cwe:
                continue
            if cwe not in cwe_tensor_map:
                cwe_tensor_map[cwe] = VulnerabilityData(
                    vulnerable=torch.tensor([]),
                    secure=torch.tensor([]),
                )

            activations = ActivationsSchema(**data)
            # print(f"Vuln ID: {activations.vuln_id}, Layer: {activations.layer}")
            vuln_tensor = torch.tensor(activations.vulnerable)
            safe_tensor = torch.tensor(activations.secure)



            vulnerable_features = torch.cat(
                (vulnerable_features, vuln_tensor.unsqueeze(0)), dim=0
            )
            safe_features = torch.cat((safe_features, safe_tensor.unsqueeze(0)), dim=0)

            cwe_tensor_map[cwe] = VulnerabilityData(
                vulnerable=torch.cat(
                    (cwe_tensor_map[cwe].vulnerable, vuln_tensor.unsqueeze(0)), dim=0
                ),
                secure=torch.cat(
                    (cwe_tensor_map[cwe].secure, safe_tensor.unsqueeze(0)), dim=0
                ),
            )


    return cwe_tensor_map[cwe]


def plot_pca(secure_features: torch.Tensor, vulnerable_features: torch.Tensor, label_secure="Secure", label_vulnerable="Vulnerable"):
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt

    # Combine features and labels
    vuln_labels = torch.ones(vulnerable_features.size(0), dtype=torch.float32)
    safe_labels = torch.zeros(secure_features.size(0), dtype=torch.float32)
    all_features = torch.cat((vulnerable_features, secure_features), dim=0).float()
    all_labels = torch.cat((vuln_labels, safe_labels), dim=0).long()

    # PCA
    pca = PCA(n_components=2)
    proj = pca.fit_transform(all_features.numpy())

    vuln_proj = proj[: len(vulnerable_features)]
    safe_proj = proj[len(vulnerable_features):]

    plt.figure(figsize=(8, 6))
    plt.scatter(
        vuln_proj[:, 0], vuln_proj[:, 1], c="r", label=label_vulnerable, alpha=0.5
    )
    plt.scatter(safe_proj[:, 0], safe_proj[:, 1], c="g", label=label_secure, alpha=0.5)
    plt.title("PCA of Activations")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.legend()
    plt.grid()
    plt.show()


def plot_lda(secure_features: torch.Tensor, vulnerable_features: torch.Tensor, label_secure="Secure", label_vulnerable="Vulnerable"):
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    import matplotlib.pyplot as plt

    # Combine features and labels
    vuln_labels = torch.ones(vulnerable_features.size(0), dtype=torch.float32)
    safe_labels = torch.zeros(secure_features.size(0), dtype=torch.float32)
    all_features = torch.cat((vulnerable_features, secure_features), dim=0).float()
    all_labels = torch.cat((vuln_labels, safe_labels), dim=0).long()

    # LDA
    lda = LinearDiscriminantAnalysis(n_components=1)
    proj = lda.fit_transform(all_features.numpy(), all_labels.numpy())

    vuln_proj = proj[: len(vulnerable_features)]
    safe_proj = proj[len(vulnerable_features):]

    plt.figure(figsize=(8, 6))
    plt.hist(vuln_proj, bins=30, alpha=0.5, label=label_vulnerable, color="r")
    plt.hist(safe_proj, bins=30, alpha=0.5, label=label_secure, color="g")
    plt.title("LDA of Activations")
    plt.xlabel("LDA Component 1")
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid()
    plt.show()


def plot_tsne(secure_features: torch.Tensor, vulnerable_features: torch.Tensor, label_secure="Secure", label_vulnerable="Vulnerable"):
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt

    # Combine features and labels
    vuln_labels = torch.ones(vulnerable_features.size(0), dtype=torch.float32)
    safe_labels = torch.zeros(secure_features.size(0), dtype=torch.float32)
    all_features = torch.cat((vulnerable_features, secure_features), dim=0).float()
    all_labels = torch.cat((vuln_labels, safe_labels), dim=0).long()

    # t-SNE
    tsne = TSNE(n_components=2, random_state=42)
    proj = tsne.fit_transform(all_features.numpy())

    vuln_proj = proj[: len(vulnerable_features)]
    safe_proj = proj[len(vulnerable_features):]

    plt.figure(figsize=(8, 6))
    plt.scatter(
        vuln_proj[:, 0], vuln_proj[:, 1], c="r", label=label_vulnerable, alpha=0.5
    )
    plt.scatter(safe_proj[:, 0], safe_proj[:, 1], c="g", label=label_secure, alpha=0.5)
    plt.title("t-SNE of Activations")
    plt.xlabel("t-SNE Component 1")
    plt.ylabel("t-SNE Component 2")
    plt.legend()
    plt.grid()
    plt.show()

for key in FisherScoreMap:
    if key == "ALL":
        continue
    logger.info(f"Layer {key[0]}, CWE {key[1]}: LDA Fisher Score = {FisherScoreMap[key]:.4f}, Probing Accuracy = {ProbingDataMap[key]:.4f}")
    layer_id = int(key[0])
    cwe = key[1]
    data = get_data(files, layer_id, cwe)
    plot_pca(data.secure, data.vulnerable, label_secure="Secure", label_vulnerable="Vulnerable")
    plot_tsne(data.secure, data.vulnerable, label_secure="Secure", label_vulnerable="Vulnerable")
    plot_lda(data.secure, data.vulnerable, label_secure="Secure", label_vulnerable="Vulnerable")

In [ ]:
plot_pca(all_safe_features["layer_0"], all_vulnerable_features["layer_0"], label_secure="Secure", label_vulnerable="Vulnerable")

plot_tsne(all_safe_features["layer_0"], all_vulnerable_features["layer_0"], label_secure="Secure", label_vulnerable="Vulnerable")


## Dump

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def plot_pca_subplots(cwe_data_map: Dict[str, VulnerabilityData]):
    """Create subplots for each CWE"""
    n_cwes = len(cwe_data_map)
    n_cols = 3
    n_rows = (n_cwes + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    axes = axes.flatten() if n_cwes > 1 else [axes]
    
    for idx, (cwe, data) in enumerate(cwe_data_map.items()):
        pca = PCA(n_components=2)
        
        # Fit PCA on combined data
        combined_data = torch.cat([data.vulnerable, data.secure], dim=0).numpy()
        pca_transformed = pca.fit_transform(combined_data)
        
        # Split back into vulnerable and safe
        vuln_size = data.vulnerable.shape[0]
        vuln_pca = pca_transformed[:vuln_size]
        safe_pca = pca_transformed[vuln_size:]
        
        ax = axes[idx]
        ax.scatter(
            vuln_pca[:, 0],
            vuln_pca[:, 1],
            c="red",
            label="Vulnerable",
            alpha=0.7,
        )
        ax.scatter(
            safe_pca[:, 0],
            safe_pca[:, 1],
            c="blue",
            label="Safe",
            alpha=0.7,
        )
        ax.set_xlabel("PCA Component 1")
        ax.set_ylabel("PCA Component 2")
        ax.set_title(f"{cwe}")
        ax.legend()
    
    # Hide unused subplots
    for idx in range(n_cwes, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle("PCA of Sparse Autoencoder Activations by CWE", fontsize=16, y=1.00)
    plt.tight_layout()
    plt.show()


# Plot subplots for each CWE
print("Plotting PCA subplots for each CWE")
plot_pca_subplots(cwe_tensor_map)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import Dict, Any

def measure_activation_distribution(activations: torch.Tensor, name: str = "Activations") -> Dict[str, Any]:
    """
    Comprehensive measurement of activation distribution statistics.
    
    Args:
        activations: Tensor of shape (n_samples, n_features)
        name: Name for the distribution (e.g., "Secure", "Vulnerable")
    
    Returns:
        Dictionary containing distribution statistics
    """
    activations_np = activations.numpy()
    
    # Flatten for overall statistics
    flat_activations = activations_np.flatten()
    
    # Sparsity analysis
    threshold = 1e-6
    sparsity = (np.abs(flat_activations) < threshold).sum() / flat_activations.size
    
    # Basic statistics
    stats_dict = {
        "name": name,
        "n_samples": activations.shape[0],
        "n_features": activations.shape[1],
        # Overall statistics
        "mean": float(np.mean(flat_activations)),
        "std": float(np.std(flat_activations)),
        "median": float(np.median(flat_activations)),
        "min": float(np.min(flat_activations)),
        "max": float(np.max(flat_activations)),
        # Distribution shape
        "skewness": float(stats.skew(flat_activations)),
        "kurtosis": float(stats.kurtosis(flat_activations)),
        # Sparsity
        "sparsity": float(sparsity),
        "active_ratio": float(1 - sparsity),
        # Percentiles
        "p25": float(np.percentile(flat_activations, 25)),
        "p75": float(np.percentile(flat_activations, 75)),
        "p95": float(np.percentile(flat_activations, 95)),
        "p99": float(np.percentile(flat_activations, 99)),
        # Per-feature statistics
        "mean_per_feature": np.mean(activations_np, axis=0),
        "std_per_feature": np.std(activations_np, axis=0),
        "max_per_feature": np.max(activations_np, axis=0),
        # Top activated features
        "top_10_features": np.argsort(np.mean(activations_np, axis=0))[-10:][::-1],
        "top_10_values": np.sort(np.mean(activations_np, axis=0))[-10:][::-1],
    }
    
    return stats_dict


def plot_activation_distribution(secure_activations: torch.Tensor, 
                                 vulnerable_activations: torch.Tensor = None,
                                 figsize=(16, 12)):
    """
    Create comprehensive visualization of activation distributions.
    
    Args:
        secure_activations: Tensor of secure code activations
        vulnerable_activations: Optional tensor of vulnerable code activations for comparison
    """
    fig, axes = plt.subplots(3, 3, figsize=figsize)
    fig.suptitle('Activation Distribution Analysis', fontsize=16, y=1.00)
    
    secure_flat = secure_activations.numpy().flatten()
    
    # 1. Overall histogram
    ax = axes[0, 0]
    ax.hist(secure_flat, bins=100, alpha=0.7, label='Secure', color='green', density=True)
    if vulnerable_activations is not None:
        vuln_flat = vulnerable_activations.numpy().flatten()
        ax.hist(vuln_flat, bins=100, alpha=0.7, label='Vulnerable', color='red', density=True)
    ax.set_xlabel('Activation Value')
    ax.set_ylabel('Density')
    ax.set_title('Overall Distribution')
    ax.legend()
    ax.set_yscale('log')
    
    # 2. Distribution without zeros (non-sparse activations)
    ax = axes[0, 1]
    secure_nonzero = secure_flat[np.abs(secure_flat) > 1e-6]
    ax.hist(secure_nonzero, bins=100, alpha=0.7, label='Secure (non-zero)', color='green', density=True)
    if vulnerable_activations is not None:
        vuln_nonzero = vuln_flat[np.abs(vuln_flat) > 1e-6]
        ax.hist(vuln_nonzero, bins=100, alpha=0.7, label='Vulnerable (non-zero)', color='red', density=True)
    ax.set_xlabel('Activation Value')
    ax.set_ylabel('Density')
    ax.set_title('Non-Zero Activations')
    ax.legend()
    
    # 3. KDE plot
    ax = axes[0, 2]
    sns.kdeplot(secure_flat, ax=ax, label='Secure', color='green', warn_singular=False)
    if vulnerable_activations is not None:
        sns.kdeplot(vuln_flat, ax=ax, label='Vulnerable', color='red', warn_singular=False)
    ax.set_xlabel('Activation Value')
    ax.set_ylabel('Density')
    ax.set_title('Kernel Density Estimate')
    ax.legend()
    
    # 4. Q-Q plot (normality test)
    ax = axes[1, 0]
    stats.probplot(secure_flat[::100], dist="norm", plot=ax)  # Subsample for speed
    ax.set_title('Q-Q Plot (Secure)')
    
    # 5. Box plot comparison
    ax = axes[1, 1]
    if vulnerable_activations is not None:
        data_to_plot = [secure_flat[::100], vuln_flat[::100]]  # Subsample
        ax.boxplot(data_to_plot, labels=['Secure', 'Vulnerable'])
    else:
        ax.boxplot([secure_flat[::100]], labels=['Secure'])
    ax.set_ylabel('Activation Value')
    ax.set_title('Box Plot Comparison')
    
    # 6. Sparsity comparison
    ax = axes[1, 2]
    secure_sparsity = (np.abs(secure_flat) < 1e-6).sum() / secure_flat.size * 100
    if vulnerable_activations is not None:
        vuln_sparsity = (np.abs(vuln_flat) < 1e-6).sum() / vuln_flat.size * 100
        ax.bar(['Secure', 'Vulnerable'], [secure_sparsity, vuln_sparsity], 
               color=['green', 'red'], alpha=0.7)
    else:
        ax.bar(['Secure'], [secure_sparsity], color=['green'], alpha=0.7)
    ax.set_ylabel('Sparsity (%)')
    ax.set_title('Activation Sparsity')
    ax.set_ylim([0, 100])
    
    # 7. Mean activation per feature (top features)
    ax = axes[2, 0]
    mean_per_feature = np.mean(secure_activations.numpy(), axis=0)
    top_n = 50
    top_indices = np.argsort(mean_per_feature)[-top_n:]
    ax.bar(range(top_n), mean_per_feature[top_indices], color='green', alpha=0.7)
    ax.set_xlabel('Feature Index (sorted)')
    ax.set_ylabel('Mean Activation')
    ax.set_title(f'Top {top_n} Most Activated Features (Secure)')
    
    # 8. Activation magnitude distribution
    ax = axes[2, 1]
    secure_magnitudes = np.linalg.norm(secure_activations.numpy(), axis=1)
    ax.hist(secure_magnitudes, bins=50, alpha=0.7, label='Secure', color='green', density=True)
    if vulnerable_activations is not None:
        vuln_magnitudes = np.linalg.norm(vulnerable_activations.numpy(), axis=1)
        ax.hist(vuln_magnitudes, bins=50, alpha=0.7, label='Vulnerable', color='red', density=True)
    ax.set_xlabel('L2 Norm')
    ax.set_ylabel('Density')
    ax.set_title('Activation Vector Magnitude')
    ax.legend()
    
    # 9. Cumulative distribution
    ax = axes[2, 2]
    secure_sorted = np.sort(np.abs(secure_flat))
    cumsum_secure = np.cumsum(secure_sorted)
    cumsum_secure = cumsum_secure / cumsum_secure[-1]
    ax.plot(np.linspace(0, 100, len(cumsum_secure)), cumsum_secure, 
            label='Secure', color='green', linewidth=2)
    if vulnerable_activations is not None:
        vuln_sorted = np.sort(np.abs(vuln_flat))
        cumsum_vuln = np.cumsum(vuln_sorted)
        cumsum_vuln = cumsum_vuln / cumsum_vuln[-1]
        ax.plot(np.linspace(0, 100, len(cumsum_vuln)), cumsum_vuln, 
                label='Vulnerable', color='red', linewidth=2)
    ax.set_xlabel('Percentile (%)')
    ax.set_ylabel('Cumulative Sum (normalized)')
    ax.set_title('Cumulative Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def print_distribution_stats(stats_dict: Dict[str, Any]):
    """Pretty print distribution statistics."""
    print(f"\n{'='*60}")
    print(f"  {stats_dict['name']} Activation Distribution Statistics")
    print(f"{'='*60}")
    print(f"Dataset Size:")
    print(f"  Samples: {stats_dict['n_samples']:,}")
    print(f"  Features: {stats_dict['n_features']:,}")
    print(f"\nBasic Statistics:")
    print(f"  Mean:   {stats_dict['mean']:>10.6f}")
    print(f"  Std:    {stats_dict['std']:>10.6f}")
    print(f"  Median: {stats_dict['median']:>10.6f}")
    print(f"  Min:    {stats_dict['min']:>10.6f}")
    print(f"  Max:    {stats_dict['max']:>10.6f}")
    print(f"\nDistribution Shape:")
    print(f"  Skewness: {stats_dict['skewness']:>8.4f}")
    print(f"  Kurtosis: {stats_dict['kurtosis']:>8.4f}")
    print(f"\nSparsity:")
    print(f"  Sparsity:     {stats_dict['sparsity']*100:>6.2f}%")
    print(f"  Active Ratio: {stats_dict['active_ratio']*100:>6.2f}%")
    print(f"\nPercentiles:")
    print(f"  25th: {stats_dict['p25']:>10.6f}")
    print(f"  75th: {stats_dict['p75']:>10.6f}")
    print(f"  95th: {stats_dict['p95']:>10.6f}")
    print(f"  99th: {stats_dict['p99']:>10.6f}")
    print(f"\nTop 10 Most Activated Features:")
    for i, (feat_idx, feat_val) in enumerate(zip(stats_dict['top_10_features'], 
                                                   stats_dict['top_10_values'])):
        print(f"  {i+1}. Feature {feat_idx:>5d}: {feat_val:>10.6f}")
    print(f"{'='*60}\n")


# Load secure activations and analyze
print("Loading secure activations...")
secure_activations = torch.load(f"../artifacts/activations/{RUN_ID}/safe_layer_0.pt")
vulnerable_activations = torch.load(f"../artifacts/activations/{RUN_ID}/vulnerable_layer_0.pt")

print(f"Secure shape: {secure_activations.shape}")
print(f"Vulnerable shape: {vulnerable_activations.shape}")

# Measure distributions
secure_stats = measure_activation_distribution(secure_activations, "Secure Code")
vulnerable_stats = measure_activation_distribution(vulnerable_activations, "Vulnerable Code")

# Print statistics
print_distribution_stats(secure_stats)
print_distribution_stats(vulnerable_stats)

# Visualize
plot_activation_distribution(secure_activations, vulnerable_activations)